# QM 640 Capstone — Step 3h: Consolidated Screening Data Health Check

One script, run any time, that checks everything we found piecemeal tonight in a single pass instead of one diagnostic per surprise:

1. Git identity configured (avoids the "Author identity unknown" error)
2. Duplicate `accession_no` rows in `screening_TO_REVIEW.csv`
3. Duplicate `accession_no` rows in `screening_worksheet.csv`
4. Duplicate `accession_no` rows in `screening_recode_sample.csv`
5. Conflicting values across duplicate copies (in any of the three files)
6. Any row still blank where it shouldn't be
7. Cross-file consistency: does `screening_worksheet.csv`'s confirmed count match `screening_TO_REVIEW.csv`'s, once both are deduplicated?
8. Low-signal flag: confirmed `Y` rows where the classifier found **zero** AI keyword matches at all - not proof of a problem, but exactly the kind of row worth a manual second look (like the Comstock/Origin Bancorp/B&G Foods rows found tonight).

**Run this any time you're unsure of your data's state** - after a manual edit, after a session restart, before merging, before running `05`/`06`. It only reads files and prints a report; it never writes anything unless you explicitly run the optional auto-fix cell at the end.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. Also sets git identity here (not just on first clone), since a runtime restart followed by a standalone git command - not the full Cell 1 - is exactly what caused tonight's "Author identity unknown" error.

In [20]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"
REPO_NAME = "QM640-WALSH-CAPSTONE"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(f"Clone failed: no .git folder found at {BASE_DIR}")

# Always (re)set identity, every session - cheap, and prevents the exact
# "Author identity unknown" error from earlier tonight
!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 4.15 KiB | 606.00 KiB/s, done.
From https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE
   a92a7c3..b6cb6a2  main       -> origin/main
Updating a92a7c3..b6cb6a2
Fast-forward
 data/raw/screening_recode_sample.csv | 330 +++++++++++++++++------------------
 1 file changed, 165 insertions(+), 165 deletions(-)
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies + configuration

In [21]:
!pip install -q pandas

import pandas as pd

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
REVIEW_FILE = os.path.join(RAW_DIR, "screening_TO_REVIEW.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")

## Cell 3 — Full health check (read-only, safe to run any time)

In [22]:
ISSUES = []  # collected across all checks, printed as one summary at the end


def safe_load(path, label):
    """Loads a CSV, or reports it's simply not created yet (expected at early
    pipeline stages) instead of crashing the whole health check."""
    if not os.path.exists(path):
        print(f"  NOT YET CREATED - {label} doesn't exist yet at this stage of "
              f"the pipeline. This is expected if you haven't reached that step yet.")
        return None
    return pd.read_csv(path)


def check_duplicates(df, name):
    dupes = df["accession_no"].value_counts()
    dupes = dupes[dupes > 1]
    if len(dupes) == 0:
        print(f"  OK - no duplicate accession_no in {name}")
        return dupes
    extra = dupes.sum() - len(dupes)
    print(f"  ISSUE - {name}: {len(dupes)} accession_no's duplicated, {extra} extra rows")
    ISSUES.append(f"{name} has {extra} duplicate rows ({len(dupes)} accession_no's affected)")
    return dupes


def check_conflicts(df, name, dupe_ids, value_cols):
    conflicts = []
    for acc_no in dupe_ids.index:
        subset = df[df["accession_no"] == acc_no]
        for col in value_cols:
            if col not in subset.columns:
                continue
            vals = subset[col].astype(str).str.upper().unique()
            if len(vals) > 1:
                conflicts.append({"accession_no": acc_no, "column": col, "values": list(vals)})
    if conflicts:
        print(f"  ISSUE - {name}: {len(conflicts)} conflicting duplicate values - NEEDS HUMAN REVIEW, not auto-fixable")
        ISSUES.append(f"{name} has {len(conflicts)} CONFLICTING duplicate values - review manually")
        return pd.DataFrame(conflicts)
    else:
        print(f"  OK - {name}: duplicates (if any) have consistent values, safe to auto-collapse")
        return pd.DataFrame()


print("=" * 70)
print("1. GIT IDENTITY")
print("=" * 70)
result = os.popen(f"git -C {BASE_DIR} config user.email").read().strip()
print(f"  {'OK' if result else 'ISSUE'} - user.email = '{result}'" if result else "  ISSUE - git user.email not set")
if not result:
    ISSUES.append("git identity not configured - commits will fail")

print()
print("=" * 70)
print("2-3. DUPLICATE CHECK: screening_worksheet.csv")
print("=" * 70)
master = safe_load(SCREENING_FILE, "screening_worksheet.csv")
if master is not None:
    master_dupes = check_duplicates(master, "screening_worksheet.csv")
    master_conflicts = check_conflicts(master, "screening_worksheet.csv", master_dupes,
                                         ["is_genuine_ai_event", "announcement_type"])
else:
    master_dupes, master_conflicts = pd.Series(dtype=int), pd.DataFrame()

print()
print("=" * 70)
print("2-3. DUPLICATE CHECK: screening_TO_REVIEW.csv")
print("=" * 70)
review = safe_load(REVIEW_FILE, "screening_TO_REVIEW.csv (created by 03d, filled in by 03e/03e2)")
if review is not None:
    review_dupes = check_duplicates(review, "screening_TO_REVIEW.csv")
    review_conflicts = check_conflicts(review, "screening_TO_REVIEW.csv", review_dupes,
                                         ["is_genuine_ai_event", "announcement_type"])
else:
    review_dupes, review_conflicts = pd.Series(dtype=int), pd.DataFrame()

print()
print("=" * 70)
print("4. DUPLICATE CHECK: screening_recode_sample.csv")
print("=" * 70)
recode = safe_load(RECODE_FILE, "screening_recode_sample.csv")
if recode is not None:
    recode_dupes = check_duplicates(recode, "screening_recode_sample.csv")
    recode_conflicts = check_conflicts(recode, "screening_recode_sample.csv", recode_dupes,
                                         ["recoder_is_genuine_ai_event", "recoder_announcement_type"])
else:
    recode_dupes, recode_conflicts = pd.Series(dtype=int), pd.DataFrame()

print()
print("=" * 70)
print("6. BLANK-VALUE CHECK")
print("=" * 70)
if review is not None:
    review_blank = review["is_genuine_ai_event"].isna() | (review["is_genuine_ai_event"].astype(str).str.strip() == "")
    print(f"  screening_TO_REVIEW.csv: {review_blank.sum()} rows still blank" +
          (" - OK, none" if review_blank.sum() == 0 else " - ISSUE, finish reviewing these first"))
    if review_blank.sum() > 0:
        ISSUES.append(f"{review_blank.sum()} rows in screening_TO_REVIEW.csv still unclassified")
else:
    print("  SKIPPED - screening_TO_REVIEW.csv doesn't exist yet")

print()
print("=" * 70)
print("7. CROSS-FILE CONSISTENCY (deduplicated view)")
print("=" * 70)
if master is not None and review is not None:
    master_dedup = master.drop_duplicates(subset=["accession_no"], keep="first")
    review_dedup = review.drop_duplicates(subset=["accession_no"], keep="first")

    master_y = (master_dedup["is_genuine_ai_event"].astype(str).str.upper() == "Y").sum()
    review_y = (review_dedup["is_genuine_ai_event"].astype(str).str.upper() == "Y").sum()
    review_ids = set(review_dedup["accession_no"])
    master_y_in_review = master_dedup[master_dedup["accession_no"].isin(review_ids) &
                                        (master_dedup["is_genuine_ai_event"].astype(str).str.upper() == "Y")]
    master_y_outside_review = master_dedup[~master_dedup["accession_no"].isin(review_ids) &
                                             (master_dedup["is_genuine_ai_event"].astype(str).str.upper() == "Y")]

    print(f"  screening_worksheet.csv confirmed Y (deduplicated): {master_y}")
    print(f"  screening_TO_REVIEW.csv confirmed Y (deduplicated): {review_y}")
    print(f"  Of master's Y rows, {len(master_y_in_review)} are inside your reviewed set, "
          f"{len(master_y_outside_review)} are OUTSIDE it (unexpected if >0)")
    if len(master_y_outside_review) > 0:
        print("  ISSUE - master has confirmed Y rows that were never in your review file:")
        print(master_y_outside_review[["accession_no", "company_name"]].head(10))
        ISSUES.append(f"{len(master_y_outside_review)} confirmed Y rows in master fall outside your reviewed set")
    if master_y != review_y:
        ISSUES.append(f"Master ({master_y}) and review file ({review_y}) confirmed counts DISAGREE - master needs re-merging from review")
else:
    review_dedup = None
    print("  SKIPPED - needs both screening_worksheet.csv and screening_TO_REVIEW.csv")

print()
print("=" * 70)
print("8. LOW-SIGNAL FLAG (zero keyword match but marked Y - worth a manual glance)")
print("=" * 70)
if review_dedup is not None and "matched_keywords" in review_dedup.columns:
    low_signal = review_dedup[(review_dedup["is_genuine_ai_event"].astype(str).str.upper() == "Y") &
                                (review_dedup["matched_keywords"].fillna("") == "")]
    print(f"  {len(low_signal)} confirmed Y rows with NO matched AI keyword at all")
    if len(low_signal) > 0:
        print("  (Not necessarily wrong - could be real content past the snippet cutoff - "
              "but worth a spot check, especially for companies outside obvious tech/AI sectors)")
        print(low_signal["company_name"].value_counts().head(10))
elif review_dedup is None:
    print("  SKIPPED - screening_TO_REVIEW.csv not available yet")
else:
    print("  Skipped - matched_keywords column not present (only exists after 03e2 has run)")

print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)
if not ISSUES:
    print("ALL CHECKS PASSED (for the files that exist at this pipeline stage). Safe to proceed.")
else:
    print(f"{len(ISSUES)} ISSUE(S) FOUND:")
    for i, issue in enumerate(ISSUES, 1):
        print(f"  {i}. {issue}")
    print("\nRun Cell 4 below to auto-fix everything EXCEPT conflicting duplicates "
          "(those need your judgment) and low-signal rows (those need a manual read).")

1. GIT IDENTITY
  OK - user.email = 'Shan_muganathan@yahoo.com'

2-3. DUPLICATE CHECK: screening_worksheet.csv
  OK - no duplicate accession_no in screening_worksheet.csv
  OK - screening_worksheet.csv: duplicates (if any) have consistent values, safe to auto-collapse

2-3. DUPLICATE CHECK: screening_TO_REVIEW.csv
  OK - no duplicate accession_no in screening_TO_REVIEW.csv
  OK - screening_TO_REVIEW.csv: duplicates (if any) have consistent values, safe to auto-collapse

4. DUPLICATE CHECK: screening_recode_sample.csv
  OK - no duplicate accession_no in screening_recode_sample.csv
  OK - screening_recode_sample.csv: duplicates (if any) have consistent values, safe to auto-collapse

6. BLANK-VALUE CHECK
  screening_TO_REVIEW.csv: 0 rows still blank - OK, none

7. CROSS-FILE CONSISTENCY (deduplicated view)
  screening_worksheet.csv confirmed Y (deduplicated): 514
  screening_TO_REVIEW.csv confirmed Y (deduplicated): 514
  Of master's Y rows, 514 are inside your reviewed set, 0 are OUTSIDE

## Cell 4 — Auto-fix (only run after reviewing Cell 3's summary)

Safely collapses duplicate rows in all three files (only where Cell 3 confirmed no conflicting values), re-merges `screening_worksheet.csv` from the deduplicated review file if the two counts disagree, and pushes. **Does not** touch conflicting duplicates or low-signal rows - those print separately above for you to handle by hand.

In [23]:
changed = []

if master is not None and len(master_dupes) > 0 and len(master_conflicts) == 0:
    master.drop_duplicates(subset=["accession_no"], keep="first").to_csv(SCREENING_FILE, index=False)
    changed.append("data/raw/screening_worksheet.csv")
    print(f"Deduplicated screening_worksheet.csv: {len(master)} -> {master['accession_no'].nunique()} rows")

if review is not None and len(review_dupes) > 0 and len(review_conflicts) == 0:
    review.drop_duplicates(subset=["accession_no"], keep="first").to_csv(REVIEW_FILE, index=False)
    changed.append("data/raw/screening_TO_REVIEW.csv")
    print(f"Deduplicated screening_TO_REVIEW.csv: {len(review)} -> {review['accession_no'].nunique()} rows")

if recode is not None and len(recode_dupes) > 0 and len(recode_conflicts) == 0:
    recode.drop_duplicates(subset=["accession_no"], keep="first").to_csv(RECODE_FILE, index=False)
    changed.append("data/raw/screening_recode_sample.csv")
    print(f"Deduplicated screening_recode_sample.csv: {len(recode)} -> {recode['accession_no'].nunique()} rows")

# Re-merge master from review if counts disagree and no conflicts block it -
# only possible once BOTH files exist
if master is not None and review is not None:
    master_check = pd.read_csv(SCREENING_FILE)
    review_check = pd.read_csv(REVIEW_FILE)
    m_y = (master_check["is_genuine_ai_event"].astype(str).str.upper() == "Y").sum()
    r_y = (review_check["is_genuine_ai_event"].astype(str).str.upper() == "Y").sum()
    if m_y != r_y and len(master_conflicts) == 0 and len(review_conflicts) == 0:
        update_cols = ["accession_no", "is_genuine_ai_event", "announcement_type"]
        updates = review_check[update_cols].drop_duplicates(subset=["accession_no"], keep="last")
        master_check = master_check.set_index("accession_no")
        updates_idx = updates.set_index("accession_no")
        master_check.loc[updates_idx.index, "is_genuine_ai_event"] = updates_idx["is_genuine_ai_event"]
        master_check.loc[updates_idx.index, "announcement_type"] = updates_idx["announcement_type"]
        master_check = master_check.reset_index()
        master_check.to_csv(SCREENING_FILE, index=False)
        changed.append("data/raw/screening_worksheet.csv")
        print(f"Re-merged screening_worksheet.csv from review file (counts had disagreed: {m_y} vs {r_y})")
else:
    print("Skipping re-merge check - needs both screening_worksheet.csv and screening_TO_REVIEW.csv to exist")

if changed:
    for f in set(changed):
        os.system(f'git -C {BASE_DIR} add "{f}"')
    os.system(f'git -C {BASE_DIR} commit -m "Step 3h: automated dedup/consistency fix"')
    os.system(f'git -C {BASE_DIR} push')
    print(f"\nCommitted and pushed: {sorted(set(changed))}")
else:
    print("\nNothing to fix - all clean.")

# Final confirmed count - only meaningful once screening_worksheet.csv exists
if master is not None:
    final_master = pd.read_csv(SCREENING_FILE)
    final_confirmed = (final_master["is_genuine_ai_event"].astype(str).str.upper() == "Y").sum()
    print(f"\nFINAL confirmed count: {final_confirmed}")
    print(final_master.loc[final_master["is_genuine_ai_event"].astype(str).str.upper() == "Y",
                            "announcement_type"].value_counts())
else:
    print("\nscreening_worksheet.csv doesn't exist yet - run 03 Part A first.")


Nothing to fix - all clean.

FINAL confirmed count: 514
announcement_type
M&A            211
R&D            199
partnership    104
Name: count, dtype: int64


## What this deliberately does NOT auto-fix

- **Conflicting duplicate values** (same `accession_no`, different `Y`/`N` or type across copies) - printed by Cell 3, requires you to pick the right answer
- **Low-signal rows** (confirmed `Y`, zero keyword match) - printed by Cell 3, requires actually reading the filing, same as the Comstock/Origin Bancorp/B&G Foods check

Those two categories are judgment calls, not data-quality bugs - no script should resolve them silently.